In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting AS (
WITH deduped_id_activities_filtered_text AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY activity_id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_email_body
    WHERE
        body_text ILIKE '%A new event has been scheduled.%'
),
latest_text_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_text
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_meeting
FROM
    latest_text_per_lead_id
WHERE
    lead_id_rn = 1  
)